In [0]:
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.MPSII_Patient_Cohort AS
WITH
/* ============================================================================
   1) ELIGIBILITY COHORT BUILD
   Goal: Identify eligible MPS II patients using:
     A) "Specified" Dx (E761) with >=2 distinct Dx dates + ANY qualifying treatment evidence
     B) "Incremental Unspecified" Dx (E763) with >=2 distinct Dx dates + Elaprase-only evidence
        and NOT already included in (A)
   ========================================================================== */

-- Pull all "Specified" diagnosis events (E761) within the diagnosis window.
MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS DX_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE AS DX_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Keep patients with >=2 distinct Dx dates for "Specified".
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT DX_DATE) >= 2
),

-- Pull all "Unspecified" diagnosis events (E763) within the diagnosis window.
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS DX_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE AS DX_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN '2020-08-01' AND (SELECT end_date FROM runtime_parameters)
),

-- Keep patients with >=2 distinct Dx dates for "Unspecified".
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT DX_DATE) >= 2
),

/* ============================================================================
   2) TREATMENT EVIDENCE
   ========================================================================== */

-- Broad treatment universe for the specified cohort.
-- Includes:
--   1) Elaprase + added NDC
--   2) standard infusion/procedure codes
--   3) J3490/J3590/J9999 only on/after 2026-03-01
MPSII_Treatment_Universe AS (

    -- 1. MEDICAL: NDC-based
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS FILL_DATE,
        NDC11 AS TX_CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001', '540920700', '8497600101')
      AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

    UNION ALL

    -- 2. PHARMACY
    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE,
        NDC11 AS TX_CODE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001', '540920700', '8497600101')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

    UNION ALL

    -- 3. STANDARD PROCEDURES
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PROCEDURE_CODE AS TX_CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366','J1743','S9357','S9379',
        '38206','38230','38232','38240','38241','38242','38243','38250'
    )
      AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

    UNION ALL

    -- 4. J-CODES WITH STRICT DATE RULE
    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS FILL_DATE,
        PROCEDURE_CODE AS TX_CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN ('J3490', 'J3590', 'J9999')
      AND SERVICE_DATE >= '2026-03-01'
),

-- Narrow treatment evidence for incremental unspecified cohort.
-- Keep this Elaprase-only logic unchanged unless you explicitly want to broaden it too.
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID
    FROM (
        SELECT DISTINCT
            PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001', '540920700')
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

        UNION ALL

        SELECT DISTINCT
            PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001', '540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

        UNION ALL

        SELECT DISTINCT
            PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)
    ) t
),

/* ============================================================================
   3) COHORT LOGIC
   ========================================================================== */

-- Eligible "Specified" = >=2 Dx dates AND any treatment evidence from treatment universe.
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT
        p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN (
        SELECT DISTINCT PATIENT_ID
        FROM MPSII_Treatment_Universe
    ) t
        ON p.PATIENT_ID = t.PATIENT_ID
),

-- Eligible "Incremental Unspecified" = >=2 Dx dates AND Elaprase-only evidence,
-- excluding anyone already in the specified+treatment set.
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT
        p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t
        ON p.PATIENT_ID = t.PATIENT_ID
    WHERE p.PATIENT_ID NOT IN (
        SELECT PATIENT_ID
        FROM Patients_2Dx_Specified_With_Treatment
    )
)

-- Final eligible patient list.
SELECT PATIENT_ID
FROM Patients_2Dx_Specified_With_Treatment

UNION

SELECT PATIENT_ID
FROM Patients_Incremental_Unspecified;

In [0]:
SELECT *
FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_Patient_Cohort;

In [0]:
CREATE OR REPLACE Table com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims AS
WITH cohort AS (
    SELECT DISTINCT PATIENT_ID
    FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_Patient_Cohort
)

-- 1) MEDICAL CLAIMS: NDC-based
SELECT DISTINCT
    m.PATIENT_ID,
    COALESCE(m.RENDERING_NPI, m.REFERRING_NPI) AS HCP_NPI,
    m.SERVICE_DATE AS CLAIM_DATE,
    'MEDICAL' AS CLAIM_SOURCE,
    'NDC' AS CODE_TYPE,
    m.NDC11 AS TX_CODE,
    CASE
        WHEN m.NDC11 IN ('54092070001', '540920700') THEN 'ELAPRASE'
        WHEN m.NDC11 = '8497600101' THEN 'AVLAYAH'
    END AS BRAND,
    m.PROCEDURE_CODE,
    m.DIAGNOSIS_CODES,
    m.MEDICAL_EVENT_ID AS CLAIM_ID
FROM com_edp_prd.com_raw.kom_medical_events m
INNER JOIN cohort c
    ON m.PATIENT_ID = c.PATIENT_ID
WHERE m.NDC11 IN ('54092070001', '540920700', '8497600101')
  AND m.SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION ALL

-- 2) PHARMACY CLAIMS: NDC-based
SELECT DISTINCT
    p.PATIENT_ID,
    p.PRESCRIBER_NPI AS HCP_NPI,
    p.FILL_DATE AS CLAIM_DATE,
    'PHARMACY' AS CLAIM_SOURCE,
    'NDC' AS CODE_TYPE,
    p.NDC11 AS TX_CODE,
    CASE
        WHEN p.NDC11 IN ('54092070001', '540920700') THEN 'ELAPRASE'
        WHEN p.NDC11 = '8497600101' THEN 'AVLAYAH'
    END AS BRAND,
    CAST(NULL AS STRING) AS PROCEDURE_CODE,
    p.DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
    p.PHARMACY_EVENT_ID AS CLAIM_ID
FROM com_edp_prd.com_raw.kom_pharmacy_events p
INNER JOIN cohort c
    ON p.PATIENT_ID = c.PATIENT_ID
WHERE p.NDC11 IN ('54092070001', '540920700', '8497600101')
  AND p.TRANSACTION_RESULT = 'PAID'
  AND p.FILL_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION ALL

-- 3) MEDICAL CLAIMS: OTHER ERT PROC
SELECT DISTINCT
    m.PATIENT_ID,
    COALESCE(m.RENDERING_NPI, m.REFERRING_NPI) AS HCP_NPI,
    m.SERVICE_DATE AS CLAIM_DATE,
    'MEDICAL' AS CLAIM_SOURCE,
    'PROC' AS CODE_TYPE,
    m.PROCEDURE_CODE AS TX_CODE,
    'OTHER ERT PROC' AS BRAND,
    m.PROCEDURE_CODE,
    m.DIAGNOSIS_CODES,
    m.MEDICAL_EVENT_ID AS CLAIM_ID
FROM com_edp_prd.com_raw.kom_medical_events m
INNER JOIN cohort c
    ON m.PATIENT_ID = c.PATIENT_ID
WHERE m.PROCEDURE_CODE IN (
    '99601','99602','96365','96366','S9357','S9379',
    '38206','38230','38232','38240','38241','38242','38243','38250'
)
  AND m.SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION ALL

-- 4) MEDICAL CLAIMS: ELAPRASE procedure code
SELECT DISTINCT
    m.PATIENT_ID,
    COALESCE(m.RENDERING_NPI, m.REFERRING_NPI) AS HCP_NPI,
    m.SERVICE_DATE AS CLAIM_DATE,
    'MEDICAL' AS CLAIM_SOURCE,
    'PROC' AS CODE_TYPE,
    m.PROCEDURE_CODE AS TX_CODE,
    'ELAPRASE' AS BRAND,
    m.PROCEDURE_CODE,
    m.DIAGNOSIS_CODES,
    m.MEDICAL_EVENT_ID AS CLAIM_ID
FROM com_edp_prd.com_raw.kom_medical_events m
INNER JOIN cohort c
    ON m.PATIENT_ID = c.PATIENT_ID
WHERE m.PROCEDURE_CODE = 'J1743'
  AND m.SERVICE_DATE BETWEEN '2023-08-01' AND (SELECT end_date FROM runtime_parameters)

UNION ALL

-- 5) MEDICAL CLAIMS: AVLAYAH J-codes with strict date rule
SELECT DISTINCT
    m.PATIENT_ID,
    COALESCE(m.RENDERING_NPI, m.REFERRING_NPI) AS HCP_NPI,
    m.SERVICE_DATE AS CLAIM_DATE,
    'MEDICAL' AS CLAIM_SOURCE,
    'PROC' AS CODE_TYPE,
    m.PROCEDURE_CODE AS TX_CODE,
    'AVLAYAH' AS BRAND,
    m.PROCEDURE_CODE,
    m.DIAGNOSIS_CODES,
    m.MEDICAL_EVENT_ID AS CLAIM_ID
FROM com_edp_prd.com_raw.kom_medical_events m
INNER JOIN cohort c
    ON m.PATIENT_ID = c.PATIENT_ID
WHERE m.PROCEDURE_CODE IN ('J3490', 'J3590', 'J9999')
  AND m.SERVICE_DATE >= '2026-03-01';

In [0]:
select distinct BRAND from com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims

In [0]:
select DATE_TRUNC('month',claim_date) , count(distinct patient_id) from com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims group by 1 order by 2 desc

In [0]:
SELECT patient_id, brand
FROM (
    SELECT patient_id,
           claim_date,
           brand,
           ROW_NUMBER() OVER (
               PARTITION BY patient_id
               ORDER BY claim_date DESC
           ) AS rn
    FROM com_edp_prd.cmpa_insights_internal_schema.MPSII_Treatment_claims
) t
WHERE rn = 1
-- group by 1 order by 2 desc;

In [0]:
select distinct coalesce(RENDERING_NPI,REFERRING_NPI) from com_edp_prd.com_raw.kom_medical_events 
where PATIENT_ID = 'CZ868S7G'
and (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
-- and NDC11 IN ('54092070001', '540920700')

In [0]:
select
com_edp_prd.com_raw.kom_providers